# Efektywne programowanie w języku Python
## Wykład 7 — obiekty, protokoły i zasoby

### Cztery niezależne bloki
1. Klasy, stan i atrybuty.
2. Protokoły, operatory i dziedziczenie.
3. Wyjątki i zarządzanie zasobami.
4. Dataclasses.

Materiał na kilka spotkań zgodnie z planem kursu. Rozszerzenia są opcjonalne. Przykłady: Python 3.10, bez dodatkowych pakietów.

In [ ]:
import sys
print("Python", sys.version.split()[0])

def expect_error(error_type, operation):
    try:
        operation()
    except error_type as error:
        print(type(error).__name__, str(error))
        return error
    raise AssertionError(f"Oczekiwano {error_type.__name__}")

## A. Klasy i stan
Nazwa wskazuje obiekt. Obiekt ma tożsamość, typ i stan.
Definicja klasy wykonuje ciało klasy i tworzy obiekt klasy; wywołanie klasy zwykle tworzy instancję.

Instancje mogą obsługiwać wiele operacji, np. wywołanie, iterowanie czy dodawanie — zależnie od protokołów ich typu.

In [ ]:
class Greeting:
    language = "pl"
    def __init__(self, name):
        self.name = name
    def greet(self):
        return f"Cześć, {self.name}!"

person = Greeting("Ada")
assert person.greet() == Greeting.greet(person) == "Cześć, Ada!"
assert person.greet.__self__ is person
assert person.greet.__func__ is Greeting.greet
print(person.greet())

### `__new__` a `__init__`
`__new__` tworzy lub zwraca obiekt. Gdy zwróci instancję odpowiedniego typu, `__init__` inicjalizuje jej stan i musi zwrócić `None`.

Zwykle wystarcza `__init__`. Nie tworzy on klasy. Obsługę nietypowego tworzenia obiektów traktujemy jako rozszerzenie.

In [ ]:
creation_events = []
class Created:
    def __new__(cls, value):
        creation_events.append("new")
        return super().__new__(cls)
    def __init__(self, value):
        creation_events.append("init")
        self.value = value

created = Created(7)
assert creation_events == ["new", "init"] and created.value == 7
print(creation_events)

### Jedna aktywna metoda `__init__` w klasie
Można napisać kilka definicji `__init__`, ale każda przypisuje funkcję do tej samej nazwy. **Ostatnia zastępuje poprzednie** — Python nie wybiera wersji na podstawie liczby ani typów argumentów.

Dotyczy to także innych metod o tej samej nazwie. To nie przeciążanie metod znane np. z Javy lub C++.

In [ ]:
class ReplacedInit:
    def __init__(self):
        self.value = 0
    def __init__(self, value):  # zastępuje wcześniejszą definicję
        self.value = value

assert ReplacedInit(7).value == 7
expect_error(TypeError, lambda: ReplacedInit())

### Różne sposoby tworzenia obiektu
Jedno `__init__` może przyjmować argumenty domyślne i nazwane. Alternatywne sposoby tworzenia obiektu warto nazwać przez `@classmethod`, np. `from_text()`.

Adnotacje typów nie wprowadzają przeciążania. `typing.overload` opisuje sygnatury dla narzędzi typujących; nadal wymaga jednej rzeczywistej implementacji.

In [ ]:
class Measurement:
    def __init__(self, value=0, *, unit="m"):
        self.value = float(value)
        self.unit = unit
    @classmethod
    def from_text(cls, text):
        value, unit = text.split()
        return cls(value, unit=unit)

assert Measurement().value == 0
assert Measurement(3, unit="cm").unit == "cm"
parsed = Measurement.from_text("2.5 m")
assert parsed.value == 2.5 and parsed.unit == "m"

### Stan klasy i instancji
Atrybut klasy może być wspólną wartością domyślną. Przypisanie atrybutu instancji może go przesłonić.

Poniższy przykład dotyczy zwykłego atrybutu, bez deskryptora. Usunięcie wpisu instancji odsłania wartość klasy.

In [ ]:
class Settings:
    limit = 10

settings = Settings()
assert settings.limit == 10
settings.limit = 3
assert settings.limit == 3 and Settings.limit == 10
del settings.limit
assert settings.limit == 10
print(vars(settings), Settings.limit)

### Dynamiczne atrybuty — nie dla każdego obiektu
W zwykłej klasie można dopisać atrybut. `__dict__` nie jest jednak gwarantowane dla wszystkich instancji, np. klas ze `__slots__`.

Nie opieraj całego modelu atrybutów na założeniu „to zawsze wpis do słownika”.

In [ ]:
settings.counter = 1
seen = []
while settings.counter < 10:
    seen.append(settings.counter)
    settings.counter *= 2
assert seen == [1, 2, 4, 8]
del settings.counter
print(seen)

### Dwie pułapki współdzielonej listy
Lista jako atrybut klasy jest wspólna. Lista jako argument domyślny także powstaje raz przy definiowaniu funkcji.

To dwa różne źródła podobnego błędu — żadne nie jest poprawką drugiego.

In [ ]:
class SharedDog:
    tricks = []
    def add_trick(self, trick):
        self.tricks.append(trick)

first, second = SharedDog(), SharedDog()
first.add_trick("siad")
assert second.tricks == ["siad"]

class DefaultDog:
    def __init__(self, tricks=[]):  # celowy antywzorzec
        self.tricks = tricks

assert DefaultDog().tricks is DefaultDog().tricks

In [ ]:
class Dog:
    def __init__(self, name, tricks=None):
        self.name = name
        self.tricks = [] if tricks is None else list(tricks)
    def add_trick(self, trick):
        self.tricks.append(trick)

fido, buddy = Dog("Fido"), Dog("Buddy")
fido.add_trick("siad")
assert buddy.tricks == []
source = ["aport"]
dog = Dog("Rex", source)
source.append("zostań")
assert dog.tricks == ["aport"]  # płytka kopia, świadomy kontrakt

### Publiczny atrybut czy `property`?
Dla prostych danych zaczynamy od atrybutu. `property` dodaje walidację lub obliczenie bez zmiany składni dostępu.

Nie tworzymy mechanicznie getterów i setterów. Długie operacje I/O i działania wymagające wielu argumentów lepiej wyrazić metodą.

In [ ]:
class Temperature:
    def __init__(self, celsius):
        self.celsius = celsius
    @property
    def celsius(self):
        return self._celsius
    @celsius.setter
    def celsius(self, value):
        if value < -273.15:
            raise ValueError("Poniżej zera absolutnego")
        self._celsius = float(value)
    @property
    def fahrenheit(self):
        return self.celsius * 9 / 5 + 32

In [ ]:
temperature = Temperature(0)
assert temperature.fahrenheit == 32
expect_error(ValueError, lambda: setattr(temperature, "celsius", -300))
assert temperature.celsius == 0
expect_error(AttributeError, lambda: setattr(temperature, "fahrenheit", 100))

### Dlaczego „najpierw `__dict__` instancji” nie wystarcza?
W standardowym wyszukiwaniu deskryptor danych klasy, np. `property`, ma pierwszeństwo przed słownikiem instancji.
Potem rozpatrywany jest wpis instancji, a następnie pozostałe atrybuty klasy według MRO.

`__getattr__` jest mechanizmem awaryjnym dla nieudanego wyszukania. Własne `__getattribute__` może zmieniać ten model.

In [ ]:
temperature.__dict__["celsius"] = 999
assert temperature.celsius == 0  # property wygrywa z tym wpisem
print(temperature.__dict__)

class Coordinates:
    __slots__ = ("x", "y")
    def __init__(self, x, y):
        self.x, self.y = x, y

coordinate = Coordinates(1, 2)
assert not hasattr(coordinate, "__dict__")
expect_error(AttributeError, lambda: setattr(coordinate, "z", 3))

### Rozszerzenie: `__getattr__` musi zachować protokół
Brak atrybutu oznacza `AttributeError`, nie `KeyError`. Od tego zależy m.in. `hasattr()` i `getattr(obj, name, default)`.

Nie przechwytujemy wszystkich przypisań, aby automatycznie zmieniać dowolne pola na `float`. Dla pojedynczego pola zwykle wystarczy `property`.

In [ ]:
class Aliases:
    def __init__(self):
        self._values = {"answer": 42}
    def __getattr__(self, name):
        try:
            return self._values[name]
        except KeyError:
            raise AttributeError(name) from None

aliases = Aliases()
assert aliases.answer == 42
assert not hasattr(aliases, "missing")
assert getattr(aliases, "missing", 0) == 0

### Niepubliczne API i name mangling
`_name` sygnalizuje element niepubliczny. `__name` w klasie uruchamia zmianę nazwy, głównie aby ograniczyć przypadkowe kolizje w dziedziczeniu.

To nie mechanizm ochrony sekretów ani uprawnienia dostępu. Nazwy `__name__` oznaczają z kolei protokoły Pythona.

In [ ]:
class Cat:
    def __init__(self, name):
        self.name = name
    def __say_name(self):
        return self.name
    def introduce(self):
        return self.__say_name()

cat = Cat("Tom")
assert cat.introduce() == "Tom"
expect_error(AttributeError, lambda: cat.__say_name())
assert cat._Cat__say_name() == "Tom"  # demonstracja, nie publiczne API

## B. Protokoły i dziedziczenie
Metody specjalne łączą obiekt ze składnią języka: `len`, `in`, operatory i iteracja.

Implementujemy tylko potrzebne protokoły i respektujemy ich wymagane wyniki. Puste `pass` nie jest działającą implementacją.

In [ ]:
class Basket:
    def __init__(self, items=()):
        self._items = list(items)
    def __len__(self):
        return len(self._items)
    def __contains__(self, item):
        return item in self._items
    def __getitem__(self, key):
        return self._items[key]
    def __iter__(self):
        return iter(self._items)
    def __repr__(self):
        return f"Basket({self._items!r})"

In [ ]:
basket = Basket(["jabłko", "banan"])
assert len(basket) == 2 and "banan" in basket
assert basket[0] == "jabłko" and basket[::-1] == ["banan", "jabłko"]
assert list(basket) == list(basket) == ["jabłko", "banan"]
assert bool(basket) and not bool(Basket())  # domyślnie korzysta z __len__
print(repr(basket))

### Rozszerzenie: `len(obj)` szuka metody na typie
Przewidź wynik po przypisaniu `obj.__len__` do pojedynczej instancji.

Niejawne wywołania wielu metod specjalnych omijają zwykłe wyszukiwanie atrybutu instancji. Dlatego bezpośrednie `obj.__len__()` nie musi zachowywać się jak `len(obj)`.

In [ ]:
class SizedExample:
    def __len__(self):
        return 3

sized = SizedExample()
sized.__len__ = lambda: 99
assert sized.__len__() == 99
assert len(sized) == 3
SizedExample.__len__ = lambda self: 7
assert len(sized) == 7
print("Bezpośrednio:", sized.__len__(), "przez len:", len(sized))

### `repr` i `str`
`repr()` służy do diagnostyki i jednoznacznego rozpoznawania obiektu. `str()` ma być czytelny dla odbiorcy.

Obie metody muszą zwracać napis. `repr` nie musi ujawniać całego stanu ani stanowić kodu odtwarzającego obiekt. Nie pokazuj w nim sekretów.

In [ ]:
class Vector:
    def __init__(self, x, y):
        self.x, self.y = x, y
    def __repr__(self):
        return f"Vector({self.x!r}, {self.y!r})"
    def __str__(self):
        return f"({self.x}, {self.y})"
    def __add__(self, other):
        if not isinstance(other, Vector):
            return NotImplemented
        return Vector(self.x + other.x, self.y + other.y)

vector = Vector(3, 4) + Vector(1, 2)
assert (vector.x, vector.y) == (4, 6)
expect_error(TypeError, lambda: vector + 1)
print(str(vector), repr(vector))

### Rozszerzenie: operator odbity `__radd__`
Co stanie się dla `amount + 10`, a co dla `10 + amount`?
Gdy lewa strona nie obsługuje operacji i zwraca `NotImplemented`, interpreter może spróbować operatora odbitego prawej strony.

Dla podklas obowiązują dodatkowe reguły pierwszeństwa. Nie wywołujemy `__radd__` ręcznie w kodzie użytkowym.

In [ ]:
class Amount:
    def __init__(self, value):
        self.value = value
    def __add__(self, other):
        if type(other) is int:
            return Amount(self.value + other)
        return NotImplemented
    def __radd__(self, other):
        if type(other) is int:
            return Amount(other + self.value)
        return NotImplemented

amount = Amount(5)
assert (amount + 10).value == (10 + amount).value == 15
assert amount.__add__("x") is NotImplemented
expect_error(TypeError, lambda: amount + "x")

### `NotImplemented` to nie `NotImplementedError`
`NotImplemented` zwrócone przez operator informuje interpreter, że ten typ argumentu nie jest obsługiwany. Pozwala spróbować odpowiedniego operatora odbitego.

`NotImplementedError` jest wyjątkiem. Nie zastępuje tego elementu protokołu operatorów.

In [ ]:
class Order:
    def __init__(self, items=()):
        self.items = list(items)
    def __add__(self, item):
        return Order([*self.items, item])
    def __iadd__(self, item):
        self.items.append(item)
        return self

order = Order(["jabłko"])
alias = order
order = order + "banan"
assert alias.items == ["jabłko"] and order is not alias
alias = order
order += "mango"
assert order is alias and alias.items == ["jabłko", "banan", "mango"]

### Równość i hashowanie
Obiekty równe muszą mieć taki sam hash. Stan decydujący o hashu nie powinien zmieniać się podczas używania obiektu jako klucza.

Definiowanie `__eq__` bez odpowiedniego `__hash__` zwykle czyni instancje niehashowalnymi. Nie włączaj hashowania „na siłę” dla mutowalnych danych.

In [ ]:
class MutablePoint:
    def __init__(self, x, y):
        self.x, self.y = x, y
    def __eq__(self, other):
        if type(other) is not type(self):
            return NotImplemented
        return (self.x, self.y) == (other.x, other.y)

assert MutablePoint(1, 2) == MutablePoint(1, 2)
expect_error(TypeError, lambda: hash(MutablePoint(1, 2)))

### Iterator a kontener iterowalny
Iterator przechowuje postęp i zwykle zużywa się jednokrotnie; jego `__iter__` zwraca `self`.
Kontener może przy każdym `iter()` tworzyć nowy iterator.

To połączenie klas z materiałem generatorów z `part03`.

In [ ]:
class ReverseIterator:
    def __init__(self, data):
        self.data, self.index = data, len(data)
    def __iter__(self):
        return self
    def __next__(self):
        if self.index == 0:
            raise StopIteration
        self.index -= 1
        return self.data[self.index]

reverse = ReverseIterator("spam")
assert list(reverse) == list("maps")
assert list(reverse) == []

In [ ]:
class ReverseSequence:
    def __init__(self, data):
        self.data = tuple(data)
    def __iter__(self):
        for index in range(len(self.data) - 1, -1, -1):
            yield self.data[index]

sequence = ReverseSequence("spam")
assert list(sequence) == list(sequence) == list("maps")
assert iter(sequence) is not iter(sequence)

### Metoda instancji, klasowa i statyczna
- Instancji: otrzymuje `self`.
- Klasowa: otrzymuje `cls`, przydatna dla alternatywnego tworzenia instancji.
- Statyczna: nie otrzymuje automatycznie żadnego z nich.

To sposób wiązania argumentów, nie ograniczenie uprawnień do mutowania przekazanych obiektów.

In [ ]:
class Pizza:
    def __init__(self, ingredients):
        self.ingredients = list(ingredients)
    @classmethod
    def margherita(cls):
        return cls(["mozzarella", "pomidory"])
    @staticmethod
    def circle_area(radius):
        import math
        return math.pi * radius**2

class SpecialPizza(Pizza):
    pass

assert type(SpecialPizza.margherita()) is SpecialPizza
assert Pizza.circle_area(0) == 0

### Dziedziczenie i `super()`
Nadpisanie metody może rozszerzać zachowanie bazowe. `super()` wyszukuje następny element według MRO.

Nie oznacza po prostu „wywołaj mojego rodzica”. Współpracujące klasy powinny zachować zgodne sygnatury i konsekwentnie przekazywać wywołanie dalej.

In [ ]:
class Root:
    def steps(self):
        return ["Root"]
class Left(Root):
    def steps(self):
        return ["Left", *super().steps()]
class Right(Root):
    def steps(self):
        return ["Right", *super().steps()]
class Diamond(Left, Right):
    pass

assert Diamond().steps() == ["Left", "Right", "Root"]
assert Diamond.__mro__ == (Diamond, Left, Right, Root, object)
print([cls.__name__ for cls in Diamond.__mro__])

### Kompozycja jako prostszy wybór
Obiekt może korzystać z innego obiektu albo funkcji zamiast dziedziczyć wariant zachowania.

Wymienna strategia formatowania nie wymaga hierarchii klas. Dziedziczenie wybieramy dla sensownej relacji typu i zgodności kontraktu.

In [ ]:
class Report:
    def __init__(self, formatter):
        self.formatter = formatter
    def render(self, rows):
        return self.formatter(rows)

plain_report = Report(lambda rows: ", ".join(rows))
line_report = Report(lambda rows: "\n".join(rows))
assert plain_report.render(["A", "B"]) == "A, B"
assert line_report.render(["A", "B"]) == "A\nB"

### Rozszerzenie: wspólny interfejs bez dziedziczenia
Funkcja może korzystać z `render()`, nie wymagając wspólnej klasy bazowej. To duck typing.

Nie sprawdzamy nazwy klasy; interesuje nas zachowanie i kontrakt metody. Zgodna nazwa bez zgodnego znaczenia nie wystarcza.

In [ ]:
class PlainView:
    def render(self):
        return "Zwykły tekst"

class HtmlView:
    def render(self):
        return "<p>Tekst HTML</p>"

def display(view):
    return view.render()

assert display(PlainView()) == "Zwykły tekst"
assert display(HtmlView()) == "<p>Tekst HTML</p>"
assert not issubclass(HtmlView, PlainView)

### Rozszerzenie: `typing.Protocol`
Protokół opisuje wymagany interfejs dla narzędzia statycznie sprawdzającego typy. Klasa nie musi po nim dziedziczyć, aby spełniać ten interfejs.

Adnotacja nie włącza automatycznej walidacji podczas wykonania. Opcjonalne `runtime_checkable` również nie sprawdza pełnej zgodności sygnatur i znaczenia metod.

In [ ]:
from typing import Protocol

class Renderable(Protocol):
    def render(self) -> str: ...

def display_typed(view: Renderable) -> str:
    return view.render()

assert display_typed(PlainView()) == "Zwykły tekst"
assert display_typed(HtmlView()) == "<p>Tekst HTML</p>"
# Zgodność adnotacji analizuje np. mypy; te asercje sprawdzają wykonanie.

## C. Wyjątki i zasoby
Oddzielamy błąd składni od wyjątku podczas wykonania.
Celowe błędy wywołujemy przez pomocnicze `expect_error`, aby całe „Run All” mogło się zakończyć.

Przechwytujemy możliwie konkretny wyjątek, blisko operacji, której dotyczy.

In [ ]:
expect_error(SyntaxError, lambda: compile("while True print('hello')", "demo", "exec"))
expect_error(ZeroDivisionError, lambda: 1 / 0)
expect_error(TypeError, lambda: "2" + 2)
expect_error(ValueError, lambda: int("abc"))

### Wejście użytkownika bez blokowania notebooka
Funkcja przyjmuje dostawcę danych i funkcję komunikatu. Normalnie może korzystać z `input` i `print`; w demonstracji podajemy z góry odpowiedzi.

Obsługujemy błędną liczbę, ale nie połykamy zakończenia wejścia i przerwania przez użytkownika.

In [ ]:
def read_int(read=input, write=print):
    while True:
        text = read("Podaj liczbę: ")
        try:
            return int(text)
        except ValueError:
            write("To nie jest liczba całkowita")

answers = iter(["abc", "42"])
messages = []
assert read_int(lambda prompt: next(answers), messages.append) == 42
assert messages == ["To nie jest liczba całkowita"]

### Dlaczego nie gołe `except:`?
Przechwytuje także `KeyboardInterrupt` i `SystemExit`, nie tylko zwykłe błędy aplikacji.
Pętla może utrudnić przerwanie programu.

`except Exception` jest węższe, ale nadal zbyt szerokie dla konwersji liczby. Tutaj właściwe jest `except ValueError`.

### `try`, `except`, `else`, `finally`
`else` działa, gdy blok `try` zakończył się bez wyjątku i bez wcześniejszego wyjścia, np. `return`.
`finally` sprząta przy normalnym wyjściu i propagacji wyjątku.

Unikamy `return` w `finally`: może przesłonić wynik lub wyjątek. Gwałtowne zakończenie procesu nie gwarantuje wykonania sprzątania.

In [ ]:
def parse_and_double(text):
    events = []
    try:
        number = int(text)
    except ValueError:
        events.append("except")
        result = None
    else:
        events.append("else")
        result = number * 2
    finally:
        events.append("finally")
    return result, events

assert parse_and_double("3") == (6, ["else", "finally"])
assert parse_and_double("x") == (None, ["except", "finally"])

### Kontrakt błędu zamiast rankingu „lepszych” funkcji
Wyjątek, `None` lub jawny wynik ze statusem to różne kontrakty.
Wybór zależy od tego, czy brak wyniku jest zwykłym przypadkiem i czy można go pomylić z poprawną wartością.

Przy zwykłym dzieleniu sensowny jest istniejący `ZeroDivisionError`; nowy wyjątek dodajemy, gdy wnosi znaczenie domenowe.

In [ ]:
class ConfigurationError(Exception):
    """Niepoprawna konfiguracja aplikacji."""

def read_positive_limit(text):
    try:
        value = int(text)
    except ValueError as error:
        raise ConfigurationError("Limit musi być liczbą") from error
    if value <= 0:
        raise ConfigurationError("Limit musi być dodatni")
    return value

error = expect_error(ConfigurationError, lambda: read_positive_limit("abc"))
assert isinstance(error.__cause__, ValueError)
assert read_positive_limit("5") == 5

### Rozszerzenie: `raise` zachowuje propagację
W bloku `except` samo `raise` ponownie zgłasza aktualny wyjątek.
`raise NewError(...) from error` dodaje jawny związek przyczynowy.

`raise ... from None` ukrywa kontekst w prezentacji tracebacku, ale nie usuwa go magicznie z obiektu wyjątku.

### Pliki: przykłady bez danych użytkownika
Używamy katalogu tymczasowego i UTF-8. Nie zakładamy obecności `poem.txt` ani nie zapisujemy `test.txt` w repozytorium.

`with` zarządza czasem życia zasobu; nie jest automatyczną transakcją ani zabezpieczeniem przed każdym błędem zapisu.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as directory:
    poem = Path(directory) / "poem.txt"
    poem.write_text("Zażółć gęślą jaźń\n", encoding="utf-8")
    with poem.open(encoding="utf-8") as handle:
        content = handle.read()
    assert handle.closed and content.startswith("Zażółć")
print(content.strip())

### Protokół `with`
Wyrażenie tworzy menedżer; `__enter__()` zwraca wartość przypisywaną po `as` — niekoniecznie sam menedżer.
Po udanym wejściu `__exit__(exc_type, exc_value, traceback)` obsługuje wyjście.

Prawdziwy wynik `__exit__` tłumi wyjątek. Jeżeli `__enter__` zgłosi błąd, odpowiadające mu `__exit__` nie jest wywoływane.

In [ ]:
class TraceContext:
    def __init__(self, events, suppress=False):
        self.events, self.suppress = events, suppress
    def __enter__(self):
        self.events.append("enter")
        return "wartość po as"
    def __exit__(self, exc_type, exc_value, traceback):
        self.events.append(exc_type.__name__ if exc_type else "normal")
        return self.suppress

events = []
with TraceContext(events, suppress=True) as value:
    assert value == "wartość po as"
    raise ValueError("celowy błąd")
assert events == ["enter", "ValueError"]

In [ ]:
events = []
def failing_block():
    with TraceContext(events):
        raise RuntimeError("propagacja")
expect_error(RuntimeError, failing_block)
assert events == ["enter", "RuntimeError"]

class FailingEnter(TraceContext):
    def __enter__(self):
        raise ValueError("błąd wejścia")

events = []
def failed_entry():
    with FailingEnter(events):
        raise AssertionError("Blok nie powinien się rozpocząć")
expect_error(ValueError, failed_entry)
assert events == []

### `contextlib.contextmanager`
Kod przed `yield` przygotowuje zasób, wartość `yield` trafia po `as`, a `finally` sprząta.

Wyjątek z bloku `with` wraca do generatora w miejscu `yield`. Nie połykamy go przypadkowo. Generator menedżera ma przekazać sterowanie dokładnie raz.

In [ ]:
from contextlib import contextmanager

@contextmanager
def managed_text(path):
    handle = Path(path).open("w", encoding="utf-8")
    try:
        yield handle
    finally:
        handle.close()

with TemporaryDirectory() as directory:
    target = Path(directory) / "test.txt"
    with managed_text(target) as handle:
        handle.write("Żółw")
    assert handle.closed
    assert target.read_text(encoding="utf-8") == "Żółw"

### Rozszerzenie: zamknięty plik nie oznacza wycofanego zapisu
Zgłośmy wyjątek po zapisaniu części danych. `with` zamknie plik, ale nie cofnie zapisu.

Zarządzanie zasobem i transakcja to różne kontrakty. W trybie `w` wcześniejsza zawartość może zostać utracona już podczas otwarcia.

In [ ]:
with TemporaryDirectory() as directory:
    partial_path = Path(directory) / "partial.txt"
    partial_path.write_text("Poprzednia zawartość", encoding="utf-8")
    try:
        with partial_path.open("w", encoding="utf-8") as partial_handle:
            partial_handle.write("Tylko początek")
            raise RuntimeError("Przerwano tworzenie dokumentu")
    except RuntimeError:
        pass
    assert partial_handle.closed
    assert partial_path.read_text(encoding="utf-8") == "Tylko początek"
    print("Plik zamknięty:", partial_handle.closed)
    print("Pozostało:", partial_path.read_text(encoding="utf-8"))

### JSON: obliczenie oddzielone od zapisu
Funkcja czyta dokument i oblicza iloraz. Nie zmienia pliku źródłowego. Niepoprawny JSON, brak klucza i dzielenie przez zero zachowują różne wyjątki.

Wynik zapisujemy do osobnego pliku. Zapis w tym przykładzie nie jest atomowy; aktualizacja ważnego dokumentu wymaga dodatkowej strategii.

In [ ]:
import json

def divide_json(source):
    with Path(source).open(encoding="utf-8") as handle:
        operation = json.load(handle)
    result = operation["numerator"] / operation["denominator"]
    return {**operation, "result": result}

with TemporaryDirectory() as directory:
    source = Path(directory) / "input.json"
    target = Path(directory) / "output.json"
    source.write_text('{"numerator": 6, "denominator": 2}', encoding="utf-8")
    original = source.read_text(encoding="utf-8")
    target.write_text(json.dumps(divide_json(source)), encoding="utf-8")
    assert json.loads(target.read_text(encoding="utf-8"))["result"] == 3
    assert source.read_text(encoding="utf-8") == original

## D. Dataclasses
`@dataclass` generuje standardowe metody, m.in. inicjalizację, reprezentację i równość.

Adnotacje opisują pola, ale nie zapewniają automatycznej walidacji typów podczas wykonania. Zwykła klasa nadal może być lepsza dla obiektu o rozbudowanym zachowaniu.

In [ ]:
from dataclasses import dataclass, field, FrozenInstanceError

@dataclass
class Position:
    name: str
    lon: float
    lat: float
    def __post_init__(self):
        if not -180 <= self.lon <= 180 or not -90 <= self.lat <= 90:
            raise ValueError("Współrzędne poza zakresem")

position = Position("Oslo", 10.8, 59.9)
assert position == Position("Oslo", 10.8, 59.9)
expect_error(ValueError, lambda: Position("Błąd", 0, 100))
print(position)

### Fabryka mutowalnych pól
`field(default_factory=list)` tworzy listę dla każdej instancji. To odpowiednik poprawionego przykładu psów.

`frozen=True` blokuje zwykłe przypisanie pól, ale nie zamraża obiektów przechowywanych wewnątrz. Nie oznacza także automatycznie bezpiecznego klucza słownika.

In [ ]:
@dataclass
class DogRecord:
    name: str
    tricks: list[str] = field(default_factory=list)

one, two = DogRecord("A"), DogRecord("B")
one.tricks.append("siad")
assert two.tricks == []

@dataclass(frozen=True)
class FrozenBag:
    items: list[str] = field(default_factory=list)

bag = FrozenBag()
bag.items.append("zmiana wewnątrz")
expect_error(FrozenInstanceError, lambda: setattr(bag, "items", []))
expect_error(TypeError, lambda: hash(bag))

### Rozszerzenie: `replace()` nie kopiuje głęboko
`dataclasses.replace()` tworzy nową instancję, przekazując wartości pól do inicjalizacji.
Dla zwykłego pola listowego może przekazać referencję do tej samej listy.

Przewidź, czy dopisanie elementu w kopii zmieni też oryginał. `frozen=True` nie zmienia tego w głęboką niemutowalność.

In [ ]:
from dataclasses import replace

original_record = DogRecord("Fido", ["siad"])
renamed_record = replace(original_record, name="Rex")
assert renamed_record is not original_record
assert renamed_record.tricks is original_record.tricks
renamed_record.tricks.append("aport")
assert original_record.tricks == ["siad", "aport"]
independent_record = replace(original_record, tricks=original_record.tricks.copy())
independent_record.tricks.append("zostań")
assert original_record.tricks == ["siad", "aport"]
# Płytka kopia wystarcza tu dla listy napisów, nie dla dowolnie zagnieżdżonych danych.

### `slots` i `kw_only` — od Pythona 3.10
`slots=True` ogranicza dynamiczne atrybuty i może zmniejszać narzut instancji; efekt zależy od klasy i dziedziczenia.
`kw_only=True` wymaga nazw argumentów przy inicjalizacji pól.

Dla klucza wybieramy stabilny stan z hashowalnymi polami. `frozen` nie jest granicą bezpieczeństwa.

In [ ]:
@dataclass(frozen=True, slots=True, kw_only=True)
class PointKey:
    x: int
    y: int

key = PointKey(x=1, y=2)
assert not hasattr(key, "__dict__")
assert {key: "punkt"}[PointKey(x=1, y=2)] == "punkt"
expect_error(TypeError, lambda: PointKey(1, 2))

### Rozszerzenie: dataclass w `match`
Wzorzec klasowy sprawdza typ i pola — to nie tworzenie nowego obiektu.
Używamy nazw pól, co jest czytelne także dla rekordów `kw_only`.

Nie każdy wzorzec oznacza porównanie wartości nazwy; przypomnij reguły wiązania z `part01`.

In [ ]:
def describe_point(point):
    match point:
        case PointKey(x=0, y=0):
            return "początek"
        case PointKey(x=x, y=y):
            return f"punkt {x}, {y}"
        case _:
            return "inny obiekt"

assert describe_point(PointKey(x=0, y=0)) == "początek"
assert describe_point(key) == "punkt 1, 2"

### Rozszerzenie: `__init_subclass__` zamiast metaklasy
Klasa bazowa może reagować na tworzenie podklas. Dla prostego rejestru nie trzeba pisać metaklasy.

Metaklasy są narzędziem zaawansowanym. Dekorator `dataclass` nie jest metaklasą.

In [ ]:
class Plugin:
    registry = {}
    def __init_subclass__(cls, *, plugin_name, **kwargs):
        super().__init_subclass__(**kwargs)
        if plugin_name in Plugin.registry:
            raise ValueError("Powtórzona nazwa wtyczki")
        Plugin.registry[plugin_name] = cls

class TextPlugin(Plugin, plugin_name="text"):
    pass

assert Plugin.registry["text"] is TextPlugin

## Pytania kontrolne
1. Dlaczego `property` może wygrać z wpisem w `__dict__`?
2. Co odróżnia `+` od mutującego `+=` w naszym przykładzie?
3. Co wskazuje `super()` w diamencie dziedziczenia?
4. Kiedy `__exit__` tłumi wyjątek?
5. Czy `frozen=True` zamraża listę w polu?

### Odpowiedzi
1. Jest deskryptorem danych i ma pierwszeństwo w standardowym wyszukiwaniu.
2. `+` tworzy nowy obiekt, nasz `__iadd__` zmienia istniejący.
3. Następny typ w MRO, nie po prostu bezpośrednią klasę bazową.
4. Kiedy zwraca wartość prawdziwą dla wyjątku z bloku.
5. Nie — przypisanie pola i mutowanie zawartości to inne operacje.

### Self reading — podstawy i protokoły
- [Klasy i name mangling](https://docs.python.org/3.10/tutorial/classes.html)
- [Model danych: metody specjalne i atrybuty](https://docs.python.org/3.10/reference/datamodel.html)
- [super](https://docs.python.org/3.10/library/functions.html#super)
- [Wyjątki](https://docs.python.org/3.10/tutorial/errors.html)
- [Instrukcja with](https://docs.python.org/3.10/reference/compound_stmts.html#the-with-statement)

### Self reading — zasoby i rekordy
- [contextlib](https://docs.python.org/3.10/library/contextlib.html)
- [dataclasses](https://docs.python.org/3.10/library/dataclasses.html)
- [Gettery i settery — materiał uzupełniający](https://realpython.com/python-getter-setter/)
- [Dataclasses — materiał uzupełniający](https://realpython.com/python-data-classes/)

Dokumentacja 3.10 odpowiada interpreterowi kursu. Nie przenosimy bez sprawdzenia przykładów z nowszych wersji.